# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fardinhossain/FlyRankAI-ML-Internship-Task/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane 2 — Refresh / Content Opportunity Scoring.**

I picked this lane because it turns the internship's core question into a concrete, useful output: a ranked queue of pages a human reviewer should look at first. It builds directly on the starter pipeline I already ran in ML-01 (baseline score + model probability + reason codes), so I am extending something I understand instead of starting from zero.

The data also shows the problem is real and large: more than half of the starter pages are declining and they hold most of the impressions, so a reviewer with limited time genuinely needs help deciding where to look first. The lane stays honest — it is decision support, not a promise that editing a page will recover its traffic.

In [1]:
# --- Code Cell 1: Setup and check dataset exists ---
import os
import sys
import subprocess
from pathlib import Path

# Repository configuration for Google Colab
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/fardinhossain/FlyRankAI-ML-Internship-Task.git"
REPO_DIR = "FlyRankAI-ML-Internship-Task"

if IN_COLAB:
    if not Path(REPO_DIR).exists():
        print("Cloning repository into Colab runtime...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    if Path.cwd().name != REPO_DIR and Path(REPO_DIR).exists():
        os.chdir(REPO_DIR)

candidates = [
    "data/raw/content_refresh_anonymized.csv",
    "../../data/raw/content_refresh_anonymized.csv",
    "../data/raw/content_refresh_anonymized.csv"
]
path = next((p for p in candidates if os.path.exists(p)), None)
print("Dataset found at:", path)
assert path is not None and os.path.exists(path), "Starter CSV not found - clone the repo first"


Dataset found at: data/raw/content_refresh_anonymized.csv


## 2. The question: decision, action, cost of a wrong call

**Research question:** Which pages should be reviewed first for refresh, expansion, protection, pruning, or monitoring — ranked by evidence, so limited reviewer time goes to the highest-opportunity pages?

**The decision it improves:** A content reviewer (or SEO lead) deciding, each week, which pages deserve attention first out of tens of thousands of candidates.

**The action someone takes:** The reviewer opens the ranked queue, reads the reason codes for a top page (e.g. `declining_with_demand`, `stale_visible_page`), investigates that page, and chooses one action: refresh, expand, protect, prune, or monitor.

**Cost of a wrong call:** Two-sided. Ranking a healthy page too high wastes scarce reviewer hours and erodes trust in the queue. Ranking a truly declining, high-traffic page too low means traffic and revenue keep leaking while nobody looks. The most expensive mistake is a wrong *prune* recommendation on a page that still has demand — which is why every score must ship with inspectable reason codes, never a bare number.

In [2]:
# --- Code Cell 2: Decision-relevant check: candidate pool size ---
import csv
import os

candidates = [
    "data/raw/content_refresh_anonymized.csv",
    "../../data/raw/content_refresh_anonymized.csv",
    "../data/raw/content_refresh_anonymized.csv"
]
path = next(p for p in candidates if os.path.exists(p))

rows = list(csv.DictReader(open(path, encoding="utf-8")))
visible_declining = [
    r for r in rows
    if r["trend_direction"] == "down" and int(r["impressions_90d"] or 0) >= 500
]

print(f"Total pages: {len(rows):,}")
print(f"Declining + visible (>=500 impressions/90d): {len(visible_declining):,}")
print("A reviewer cannot eyeball ~10k candidates - ranking is the whole point.")


Total pages: 30,000
Declining + visible (>=500 impressions/90d): 9,961
A reviewer cannot eyeball ~10k candidates - ranking is the whole point.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [3]:
# --- Code Cell 3: 2-3 real numbers that justify this lane ---
import csv
import os

candidates = [
    "data/raw/content_refresh_anonymized.csv",
    "../../data/raw/content_refresh_anonymized.csv",
    "../data/raw/content_refresh_anonymized.csv"
]
path = next(p for p in candidates if os.path.exists(p))

rows = list(csv.DictReader(open(path, encoding="utf-8")))
n = len(rows)

down = [r for r in rows if r["trend_direction"] == "down"]
pct_down = 100 * len(down) / n

down_big = [r for r in down if int(r["impressions_90d"] or 0) >= 500]

imp_down = sum(int(r["impressions_90d"] or 0) for r in down)
imp_all = sum(int(r["impressions_90d"] or 0) for r in rows)
share_imp = 100 * imp_down / imp_all

print(f"1) {n:,} pages in the starter data; {len(down):,} are declining ({pct_down:.1f}%).")
print(f"2) {len(down_big):,} declining pages are also visible (>=500 impressions/90d) - "
      f"{100*len(down_big)/n:.1f}% of all pages need triage.")
print(f"3) Declining pages hold {share_imp:.1f}% of all 90-day impressions - "
      "the opportunity (and the leak) is concentrated where it hurts most.")


1) 30,000 pages in the starter data; 16,262 are declining (54.2%).
2) 9,961 declining pages are also visible (>=500 impressions/90d) - 33.2% of all pages need triage.
3) Declining pages hold 51.3% of all 90-day impressions - the opportunity (and the leak) is concentrated where it hurts most.


## 4. Careful words: what I can and can't claim

**What my work will be able to say:**
- *Observed:* "Pages with these measured signals (declining trend + high impressions + stale update date) ranked highest in our queue on this data snapshot."
- *Directional:* "A learned ranking beat fixed rules on precision@50 in the starter slice (0.74 vs 0.24), which suggests learning the ranking is worth trying at larger scale."
- *Decision-support:* "This queue tells a reviewer where to look first. It does not tell them what will happen if they edit the page."

**What it will never claim:**
- *Causal proof:* I cannot say "refreshing this page *will* recover its traffic" — that needs an experiment, and this data is observational.
- *Algorithm claims:* I will not claim to have found "a Google ranking factor." I found associations in scrambled, anonymized measurements.
- *Guarantees:* A high score means "review first," never "this page is guaranteed to recover."
- *Anything about real sites, people, or queries:* The data is pseudonymized, and my outputs must stay that way.

In [4]:
# --- Code Cell 4: Safety check: confirm no raw PII-like columns ---
import csv
import os

candidates = [
    "data/raw/content_refresh_anonymized.csv",
    "../../data/raw/content_refresh_anonymized.csv",
    "../data/raw/content_refresh_anonymized.csv"
]
path = next(p for p in candidates if os.path.exists(p))

cols = next(csv.reader(open(path, encoding="utf-8")))
banned = [c for c in cols if any(k in c.lower() for k in ("url", "domain", "query_text", "title_raw", "client_name"))]

print("Columns scanned:", len(cols))
print("PII-like columns found:", banned if banned else "none - safe to proceed")


Columns scanned: 44
PII-like columns found: none - safe to proceed


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.